# 4. Dynamic Programming

Bu notebook, Sutton & Barto kitabının 4. bölümünü kapsar.

## İçindekiler
1. DP'ye Giriş
2. Policy Evaluation
3. Policy Improvement
4. Policy Iteration
5. Value Iteration
6. Asynchronous DP

## 4.1 Dynamic Programming Nedir?

**Dynamic Programming (DP)**, optimal policy hesaplamak için kullanılan bir algoritma ailesidir.

### DP'nin Gereksinimleri
- Environment'ın **tam modeli** bilinmeli: $p(s', r | s, a)$
- Bu nedenle DP **model-based** bir yöntemdir

### DP'nin Avantajları
- Matematiksel olarak **optimal** çözüm garantisi
- Diğer RL metodlarının **temelini** oluşturur

### DP'nin Dezavantajları
- Model bilinmeli (çoğu zaman bilinmez)
- Büyük state space'lerde **computationally expensive**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List

class GridWorld:
    """4x4 Grid World for DP examples."""
    
    def __init__(self):
        self.rows = 4
        self.cols = 4
        self.n_states = 16
        self.n_actions = 4  # up, right, down, left
        self.terminal_states = [0, 15]
        
        self.actions = {
            0: (-1, 0),  # up
            1: (0, 1),   # right
            2: (1, 0),   # down
            3: (0, -1)   # left
        }
        self.action_symbols = ['↑', '→', '↓', '←']
    
    def state_to_pos(self, s):
        return s // self.cols, s % self.cols
    
    def pos_to_state(self, row, col):
        return row * self.cols + col
    
    def step(self, state, action):
        """Returns (next_state, reward, done)."""
        if state in self.terminal_states:
            return state, 0, True
        
        row, col = self.state_to_pos(state)
        drow, dcol = self.actions[action]
        
        new_row = max(0, min(self.rows - 1, row + drow))
        new_col = max(0, min(self.cols - 1, col + dcol))
        
        next_state = self.pos_to_state(new_row, new_col)
        reward = -1
        done = next_state in self.terminal_states
        
        return next_state, reward, done
    
    def get_transitions(self, state, action):
        """Returns list of (probability, next_state, reward, done)."""
        next_state, reward, done = self.step(state, action)
        return [(1.0, next_state, reward, done)]  # Deterministic

env = GridWorld()
print(f"Grid: {env.rows}x{env.cols}")
print(f"Terminal states: {env.terminal_states}")

In [ ]:
def plot_values(env, V, title="State Values"):
    """Value function'ı görselleştir."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    V_grid = V.reshape(env.rows, env.cols)
    
    im = ax.imshow(V_grid, cmap='RdYlGn')
    
    for i in range(env.rows):
        for j in range(env.cols):
            state = env.pos_to_state(i, j)
            color = 'white' if abs(V_grid[i, j]) > 7 else 'black'
            ax.text(j, i, f'{V_grid[i, j]:.1f}', ha='center', va='center', 
                   fontsize=16, color=color, fontweight='bold')
    
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=14)
    plt.colorbar(im)
    plt.show()

def plot_policy(env, policy, V=None, title="Policy"):
    """Policy'yi görselleştir."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    for s in range(env.n_states):
        row, col = env.state_to_pos(s)
        
        if s in env.terminal_states:
            color = 'lightgreen'
            text = 'T'
        else:
            color = 'white'
            text = env.action_symbols[policy[s]]
        
        rect = plt.Rectangle((col, env.rows - 1 - row), 1, 1,
                              facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        
        ax.text(col + 0.5, env.rows - row - 0.5, text,
               ha='center', va='center', fontsize=24, fontweight='bold')
        
        if V is not None:
            ax.text(col + 0.1, env.rows - row - 0.1, f'{V[s]:.1f}',
                   ha='left', va='top', fontsize=10, color='gray')
    
    ax.set_xlim(0, env.cols)
    ax.set_ylim(0, env.rows)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14)
    plt.show()

## 4.2 Policy Evaluation (Prediction)

Verilen bir policy $\pi$ için value function $V^\pi$ hesapla.

### Iterative Policy Evaluation

Bellman expectation equation'ı **iteratif** olarak uygula:

$$V_{k+1}(s) = \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)[r + \gamma V_k(s')]$$

Bu işlem $V_k \rightarrow V^\pi$ olarak **yakınsar** (converge eder).

In [ ]:
def policy_evaluation(env, policy, gamma=1.0, theta=1e-8):
    """
    Iterative Policy Evaluation.
    
    Args:
        env: Environment
        policy: Array of shape [n_states], action for each state
        gamma: Discount factor
        theta: Convergence threshold
    
    Returns:
        V: Value function array
    """
    V = np.zeros(env.n_states)
    
    iteration = 0
    history = [V.copy()]
    
    while True:
        delta = 0
        
        for s in range(env.n_states):
            if s in env.terminal_states:
                continue
            
            v = V[s]
            a = policy[s]
            
            # Bellman expectation update
            new_v = 0
            for prob, next_s, reward, done in env.get_transitions(s, a):
                new_v += prob * (reward + gamma * V[next_s])
            
            V[s] = new_v
            delta = max(delta, abs(v - V[s]))
        
        iteration += 1
        history.append(V.copy())
        
        if delta < theta:
            break
    
    print(f"Policy Evaluation converged in {iteration} iterations")
    return V, history

In [ ]:
# Random policy (her state'te rastgele action)
random_policy = np.random.randint(0, env.n_actions, size=env.n_states)

# Daha anlamlı: uniform random policy simulation
# Her action 0.25 olasılık - bunu simulate etmek için
def evaluate_uniform_policy(env, gamma=1.0, theta=1e-8):
    """Uniform random policy evaluation."""
    V = np.zeros(env.n_states)
    action_prob = 1.0 / env.n_actions
    
    iteration = 0
    while True:
        delta = 0
        
        for s in range(env.n_states):
            if s in env.terminal_states:
                continue
            
            v = V[s]
            new_v = 0
            
            for a in range(env.n_actions):
                for prob, next_s, reward, done in env.get_transitions(s, a):
                    new_v += action_prob * prob * (reward + gamma * V[next_s])
            
            V[s] = new_v
            delta = max(delta, abs(v - V[s]))
        
        iteration += 1
        if delta < theta:
            break
    
    print(f"Converged in {iteration} iterations")
    return V

V_uniform = evaluate_uniform_policy(env)
plot_values(env, V_uniform, "V(s) for Uniform Random Policy")

## 4.3 Policy Improvement

Mevcut value function $V^\pi$'den **daha iyi** bir policy $\pi'$ oluştur.

### Policy Improvement Theorem

Eğer tüm $s$ için:
$$Q^\pi(s, \pi'(s)) \geq V^\pi(s)$$

O zaman:
$$V^{\pi'}(s) \geq V^\pi(s)$$

### Greedy Policy

$$\pi'(s) = \arg\max_a Q^\pi(s, a) = \arg\max_a \sum_{s',r} p(s',r|s,a)[r + \gamma V^\pi(s')]$$

In [ ]:
def policy_improvement(env, V, gamma=1.0):
    """
    V'ye göre greedy policy oluştur.
    
    Returns:
        policy: Improved policy
        policy_stable: Policy değişti mi?
    """
    policy = np.zeros(env.n_states, dtype=int)
    
    for s in range(env.n_states):
        if s in env.terminal_states:
            continue
        
        # Her action için Q(s,a) hesapla
        q_values = np.zeros(env.n_actions)
        
        for a in range(env.n_actions):
            for prob, next_s, reward, done in env.get_transitions(s, a):
                q_values[a] += prob * (reward + gamma * V[next_s])
        
        # Greedy action
        policy[s] = np.argmax(q_values)
    
    return policy

# Uniform policy'nin V'sine göre improve et
improved_policy = policy_improvement(env, V_uniform)
plot_policy(env, improved_policy, V_uniform, "Improved Policy (from uniform)")

## 4.4 Policy Iteration

**Policy Evaluation** ve **Policy Improvement**'ı sırayla uygula:

$$\pi_0 \xrightarrow{E} V^{\pi_0} \xrightarrow{I} \pi_1 \xrightarrow{E} V^{\pi_1} \xrightarrow{I} \pi_2 \xrightarrow{E} ... \xrightarrow{I} \pi_* \xrightarrow{E} V^*$$

Bu işlem **sonlu** MDP'lerde her zaman **optimal policy**'ye yakınsar.

In [ ]:
def policy_iteration(env, gamma=1.0, theta=1e-8):
    """
    Policy Iteration algoritması.
    
    Returns:
        policy: Optimal policy
        V: Optimal value function
        history: Iteration history
    """
    # Random initialization
    policy = np.random.randint(0, env.n_actions, size=env.n_states)
    
    history = []
    iteration = 0
    
    while True:
        # 1. Policy Evaluation
        V, _ = policy_evaluation(env, policy, gamma, theta)
        
        # 2. Policy Improvement
        new_policy = policy_improvement(env, V, gamma)
        
        history.append({
            'iteration': iteration,
            'policy': policy.copy(),
            'V': V.copy()
        })
        
        # Check convergence
        if np.array_equal(policy, new_policy):
            print(f"Policy Iteration converged in {iteration + 1} iterations")
            break
        
        policy = new_policy
        iteration += 1
    
    return policy, V, history

optimal_policy, V_star, pi_history = policy_iteration(env)

In [ ]:
# Sonuçları görselleştir
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Value function
ax = axes[0]
V_grid = V_star.reshape(env.rows, env.cols)
im = ax.imshow(V_grid, cmap='RdYlGn')
for i in range(env.rows):
    for j in range(env.cols):
        color = 'white' if abs(V_grid[i, j]) > 1 else 'black'
        ax.text(j, i, f'{V_grid[i, j]:.1f}', ha='center', va='center',
               fontsize=16, color=color, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
ax.set_title('Optimal V*(s)', fontsize=14)

# Policy
ax = axes[1]
for s in range(env.n_states):
    row, col = env.state_to_pos(s)
    color = 'lightgreen' if s in env.terminal_states else 'lightblue'
    rect = plt.Rectangle((col, env.rows - 1 - row), 1, 1,
                          facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    
    if s not in env.terminal_states:
        ax.text(col + 0.5, env.rows - row - 0.5, env.action_symbols[optimal_policy[s]],
               ha='center', va='center', fontsize=28, fontweight='bold')
    else:
        ax.text(col + 0.5, env.rows - row - 0.5, 'T',
               ha='center', va='center', fontsize=20, fontweight='bold')

ax.set_xlim(0, env.cols)
ax.set_ylim(0, env.rows)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Optimal Policy π*', fontsize=14)

plt.tight_layout()
plt.show()

## 4.5 Value Iteration

Policy Iteration'da her seferinde **tam** policy evaluation yapmak yerine, sadece **bir sweep** yap.

### Value Iteration Update

$$V_{k+1}(s) = \max_a \sum_{s',r} p(s',r|s,a)[r + \gamma V_k(s')]$$

Bu, Bellman **optimality** equation'ın iteratif uygulamasıdır.

In [ ]:
def value_iteration(env, gamma=1.0, theta=1e-8):
    """
    Value Iteration algoritması.
    
    Returns:
        policy: Optimal policy
        V: Optimal value function
        history: Value function at each iteration
    """
    V = np.zeros(env.n_states)
    history = [V.copy()]
    
    iteration = 0
    while True:
        delta = 0
        
        for s in range(env.n_states):
            if s in env.terminal_states:
                continue
            
            v = V[s]
            
            # Max over all actions
            q_values = np.zeros(env.n_actions)
            for a in range(env.n_actions):
                for prob, next_s, reward, done in env.get_transitions(s, a):
                    q_values[a] += prob * (reward + gamma * V[next_s])
            
            V[s] = np.max(q_values)
            delta = max(delta, abs(v - V[s]))
        
        iteration += 1
        history.append(V.copy())
        
        if delta < theta:
            break
    
    # Extract policy
    policy = policy_improvement(env, V, gamma)
    
    print(f"Value Iteration converged in {iteration} iterations")
    return policy, V, history

vi_policy, vi_V, vi_history = value_iteration(env)

In [ ]:
# Value Iteration convergence
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

iterations_to_show = [0, 1, 2, 3, 5, len(vi_history)-1]

for idx, (ax, it) in enumerate(zip(axes.flat, iterations_to_show)):
    V_it = vi_history[it].reshape(env.rows, env.cols)
    im = ax.imshow(V_it, cmap='RdYlGn', vmin=-14, vmax=0)
    
    for i in range(env.rows):
        for j in range(env.cols):
            ax.text(j, i, f'{V_it[i, j]:.1f}', ha='center', va='center',
                   fontsize=12, fontweight='bold')
    
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'Iteration {it}', fontsize=12)

plt.suptitle('Value Iteration: Convergence', fontsize=14)
plt.tight_layout()
plt.show()

## 4.6 Policy Iteration vs Value Iteration

| Özellik | Policy Iteration | Value Iteration |
|---------|-----------------|----------------|
| Her iterasyon | Full evaluation + 1 improvement | 1 Bellman optimality update |
| İterasyon sayısı | Az | Çok |
| İterasyon maliyeti | Yüksek | Düşük |
| Toplam maliyet | Genelde benzer | Genelde benzer |

In [ ]:
# Karşılaştırma
print("Policy Iteration vs Value Iteration")
print("="*40)
print(f"\nPolicy Iteration: {len(pi_history)} policy iterations")
print(f"Value Iteration: {len(vi_history)-1} value iterations")

# Aynı sonuca ulaştıklarını doğrula
print(f"\nSame optimal policy: {np.array_equal(optimal_policy, vi_policy)}")
print(f"Same optimal V (approx): {np.allclose(V_star, vi_V)}")

## 4.7 Gambler's Problem (Örnek)

Klasik bir DP örneği:
- Kumarbaz $p$ olasılıkla yazı gelen bir para atıyor
- 100\$ kazanırsa oyun biter (kazandı)
- 0\$ olursa oyun biter (kaybetti)
- Her turda, mevcut parasının bir kısmını bahis olarak koyar

In [ ]:
def gamblers_problem(p_heads=0.4, gamma=1.0, theta=1e-9):
    """Gambler's Problem with Value Iteration."""
    
    goal = 100
    V = np.zeros(goal + 1)
    V[goal] = 1.0  # Win state
    
    history = [V.copy()]
    
    while True:
        delta = 0
        
        for s in range(1, goal):
            v = V[s]
            
            # Possible stakes: 1 to min(s, goal-s)
            max_stake = min(s, goal - s)
            action_values = []
            
            for stake in range(1, max_stake + 1):
                # Win: s + stake, Lose: s - stake
                value = p_heads * V[s + stake] + (1 - p_heads) * V[s - stake]
                action_values.append(value)
            
            V[s] = max(action_values)
            delta = max(delta, abs(v - V[s]))
        
        history.append(V.copy())
        
        if delta < theta:
            break
    
    # Extract policy
    policy = np.zeros(goal + 1, dtype=int)
    for s in range(1, goal):
        max_stake = min(s, goal - s)
        action_values = []
        
        for stake in range(1, max_stake + 1):
            value = p_heads * V[s + stake] + (1 - p_heads) * V[s - stake]
            action_values.append(value)
        
        # Ties: en küçük stake'i tercih et
        policy[s] = np.argmax(action_values) + 1
    
    return V, policy, history

V_gambler, policy_gambler, _ = gamblers_problem(p_heads=0.4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Value function
axes[0].plot(V_gambler, 'b-', linewidth=2)
axes[0].set_xlabel('Capital ($)')
axes[0].set_ylabel('Value V(s)')
axes[0].set_title("Gambler's Problem: Value Function (p=0.4)")
axes[0].grid(True, alpha=0.3)

# Policy
axes[1].bar(range(len(policy_gambler)), policy_gambler, color='steelblue', alpha=0.7)
axes[1].set_xlabel('Capital ($)')
axes[1].set_ylabel('Stake ($)')
axes[1].set_title("Gambler's Problem: Optimal Policy (p=0.4)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Özet

| Algoritma | Kullanım | Gereksinim |
|-----------|----------|------------|
| **Policy Evaluation** | V^π hesapla | Policy + Model |
| **Policy Improvement** | π'den daha iyi π' bul | V^π + Model |
| **Policy Iteration** | Optimal π* bul | Model |
| **Value Iteration** | Optimal V* bul | Model |

### Önemli Noktalar
- DP **tam model** gerektirir
- Her iki algoritma da **optimal**'e yakınsar
- Büyük state space'lerde **impractical** olabilir

### Sonraki Notebook
**05 - Monte Carlo Methods**: Model-free prediction and control